In [10]:
# i need the schema of interactions table
print(df_active.schema)
print(df_active.head(5))


Schema({'author': String, 'subreddit': String, 'count_posts': Int64, 'count_comments': Int64, 'interaction_count': Int64})
shape: (5, 5)
┌─────────────────┬────────────────────┬─────────────┬────────────────┬───────────────────┐
│ author          ┆ subreddit          ┆ count_posts ┆ count_comments ┆ interaction_count │
│ ---             ┆ ---                ┆ ---         ┆ ---            ┆ ---               │
│ str             ┆ str                ┆ i64         ┆ i64            ┆ i64               │
╞═════════════════╪════════════════════╪═════════════╪════════════════╪═══════════════════╡
│ Joannamoody-634 ┆ politics           ┆ 0           ┆ 4              ┆ 4                 │
│ jcowan99        ┆ politics           ┆ 0           ┆ 1              ┆ 1                 │
│ sgent           ┆ futurology         ┆ 0           ┆ 1              ┆ 1                 │
│ omglink         ┆ unexpected         ┆ 0           ┆ 1              ┆ 1                 │
│ WrenchWanderer  ┆ contagiouslaugh

In [2]:
import polars as pl
import numpy as np
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

In [3]:
# Matrix and item–item cosine

In [4]:
# Without cleaning the outliers
# ANALYTICS_DIR = Path("analytics")
# ACTIVE_PATH = ANALYTICS_DIR / "user_subreddit_active_interactions.parquet"

# df_active = pl.read_parquet(ACTIVE_PATH)
# print(df_active.shape)
# print(df_active.head())

(21061135, 5)
shape: (5, 5)
┌──────────────────────┬───────────────────┬─────────────┬────────────────┬───────────────────┐
│ author               ┆ subreddit         ┆ count_posts ┆ count_comments ┆ interaction_count │
│ ---                  ┆ ---               ┆ ---         ┆ ---            ┆ ---               │
│ str                  ┆ str               ┆ i64         ┆ i64            ┆ i64               │
╞══════════════════════╪═══════════════════╪═════════════╪════════════════╪═══════════════════╡
│ Important_Trash_4555 ┆ povertyfinance    ┆ 0           ┆ 16             ┆ 16                │
│ Shatter_starx        ┆ pcmasterrace      ┆ 0           ┆ 22             ┆ 22                │
│ Japparbyn            ┆ interestingasfuck ┆ 0           ┆ 1              ┆ 1                 │
│ Wonderful_Price2355  ┆ mademesmile       ┆ 0           ┆ 9              ┆ 9                 │
│ BeyondStars_ThenMore ┆ facepalm          ┆ 0           ┆ 7              ┆ 7                 │
└───────────

In [12]:
# cleaned outliers
ANALYTICS_DIR = Path("analytics")
ACTIVE_PATH = ANALYTICS_DIR / "clean_df_all.parquet"

df_active = pl.read_parquet(ACTIVE_PATH)
print(df_active.shape)
print(df_active.head())

(39981748, 5)
shape: (5, 5)
┌──────────────────────┬───────────────────┬─────────────┬────────────────┬───────────────────┐
│ author               ┆ subreddit         ┆ count_posts ┆ count_comments ┆ interaction_count │
│ ---                  ┆ ---               ┆ ---         ┆ ---            ┆ ---               │
│ str                  ┆ str               ┆ i64         ┆ i64            ┆ i64               │
╞══════════════════════╪═══════════════════╪═════════════╪════════════════╪═══════════════════╡
│ Important_Trash_4555 ┆ povertyfinance    ┆ 0           ┆ 16             ┆ 16                │
│ geeriveting          ┆ reddeadredemption ┆ 0           ┆ 1              ┆ 1                 │
│ Japparbyn            ┆ interestingasfuck ┆ 0           ┆ 1              ┆ 1                 │
│ BlackAbsol44         ┆ pcmasterrace      ┆ 1           ┆ 0              ┆ 1                 │
│ One-Personality3513  ┆ houseofthedragon  ┆ 2           ┆ 1              ┆ 3                 │
└───────────

In [13]:
# 1.1  user and subreddit index mappings

users = df_active.select("author").unique().to_series().to_list()
subs = df_active.select("subreddit").unique().to_series().to_list()

user_to_idx = {u: i for i, u in enumerate(users)}
sub_to_idx = {s: j for j, s in enumerate(subs)}

n_users = len(users)
n_items = len(subs)
print("[R1] n_users, n_items:", n_users, n_items)

[R1] n_users, n_items: 13094697 500


In [14]:
# 1.2  sparse user–subreddit matrix (URM)

df_idx = df_active.select(
    pl.col("author").replace(user_to_idx).alias("u"),
    pl.col("subreddit").replace(sub_to_idx).alias("i"),
    pl.col("interaction_count").alias("cnt"),
)

rows = df_idx["u"].to_numpy()
cols = df_idx["i"].to_numpy()
data = df_idx["cnt"].cast(pl.Float32).to_numpy()

URM = csr_matrix((data, (rows, cols)), shape=(n_users, n_items))
print("[R2] URM shape:", URM.shape, "nnz:", URM.nnz)

[R2] URM shape: (13094697, 500) nnz: 39981748


In [15]:
# 1.3 item–item cosine similarity

ITEMS = normalize(URM, norm="l2", axis=0, copy=True)

item_cosine = (ITEMS.T @ ITEMS).tocsr()

print("[R3] item_cosine shape:", item_cosine.shape)

[R3] item_cosine shape: (500, 500)


In [16]:
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize


In [17]:
# item–item CF + Precision@K + Recall@K + popularity baseline

class ItemItemEvaluator:
    def __init__(self, URM: csr_matrix):
        self.URM = URM.tocsr()
        self.n_users, self.n_items = self.URM.shape

        items = normalize(self.URM, norm="l2", axis=0, copy=True)
        self.item_cosine = (items.T @ items).tocsr()

        self.user_items = None
        self.train_items = None
        self.test_items = None

        # popularity baseline
        self.item_pop_counts = np.asarray(self.URM.sum(axis=0)).ravel()
        self.pop_ranking = np.argsort(-self.item_pop_counts)

    def build_user_items(self):
        coo = self.URM.tocoo()
        user_items = [[] for _ in range(self.n_users)]
        for u, i in zip(coo.row, coo.col):
            user_items[u].append(i)
        self.user_items = [np.unique(np.array(lst, dtype=np.int32)) for lst in user_items]

    def train_test_split(self, test_frac=0.1, seed=42):
        rng = np.random.default_rng(seed)
        self.train_items = []
        self.test_items = []
        for items_u in self.user_items:
            if items_u.size < 2:
                self.train_items.append(np.array([], dtype=np.int32))
                self.test_items.append(np.array([], dtype=np.int32))
                continue
            n_test = max(1, int(test_frac * items_u.size))
            test_idx = rng.choice(items_u.size, size=n_test, replace=False)
            mask = np.ones(items_u.size, dtype=bool)
            mask[test_idx] = False
            self.train_items.append(items_u[mask])
            self.test_items.append(items_u[test_idx])

    def _score_item_item(self, train_item_ids, exclude_items=None, top_k=50):
        if train_item_ids.size == 0:
            return np.array([], dtype=np.int32)
        sim_vec = np.asarray(self.item_cosine[train_item_ids].sum(axis=0)).ravel()
        if exclude_items:
            sim_vec[list(exclude_items)] = 0.0
        if top_k >= sim_vec.size:
            idx = np.argsort(-sim_vec)
        else:
            idx = np.argpartition(-sim_vec, top_k)[:top_k]
            idx = idx[np.argsort(-sim_vec[idx])]
        return idx

    @staticmethod
    def precision_at_k(recommended, relevant, k):
        if len(relevant) == 0:
            return np.nan
        topk = recommended[:k]
        hits = len(set(topk) & set(relevant))
        return hits / float(k)

    @staticmethod
    def recall_at_k(recommended, relevant, k):
        if len(relevant) == 0:
            return np.nan
        topk = recommended[:k]
        hits = len(set(topk) & set(relevant))
        return hits / float(len(relevant))

    def eval_item_item(self, K=10):
        precs, recs = [], []
        for u in range(self.n_users):
            train_u = self.train_items[u]
            test_u = self.test_items[u]
            if train_u.size == 0 or test_u.size == 0:
                continue
            rec = self._score_item_item(train_u, exclude_items=set(train_u), top_k=K)
            p = self.precision_at_k(rec, test_u, K)
            r = self.recall_at_k(rec, test_u, K)
            if not np.isnan(p):
                precs.append(p)
                recs.append(r)
        return float(np.mean(precs)) if precs else float("nan"), \
               float(np.mean(recs)) if recs else float("nan")

    def eval_popularity(self, K=10):
        precs, recs = [], []
        rec = self.pop_ranking  # same list for all users
        for u in range(self.n_users):
            test_u = self.test_items[u]
            if test_u.size == 0:
                continue
            p = self.precision_at_k(rec, test_u, K)
            r = self.recall_at_k(rec, test_u, K)
            if not np.isnan(p):
                precs.append(p)
                recs.append(r)
        return float(np.mean(precs)) if precs else float("nan"), \
               float(np.mean(recs)) if recs else float("nan")


# Comments

| Dataset Condition | Model        | Precision@10 | Recall@10 |
|------------------|-------------|-------------|-----------|
| Before Cleaning  | Item–item   | 0.0384      | 0.2578    |
| Before Cleaning  | Popularity  | 0.0192      | 0.1488    |
| After Removing Top 2% | Item–item   | 0.0292      | 0.2788    |
| After Removing Top 2% | Popularity  | 0.0206      | 0.2000    |

Removing the top 2% most active users slightly reduced precision but increased recall. This suggests that extremely active accounts contributed strong but potentially noisy co-occurrence signals. After filtering these users, the model produced broader recommendations that captured a larger portion of users’ interactions.

In [18]:
evaluator = ItemItemEvaluator(URM)
evaluator.build_user_items()
evaluator.train_test_split(test_frac=0.1, seed=42)

K = 10
p_item, r_item = evaluator.eval_item_item(K=K)
p_pop, r_pop = evaluator.eval_popularity(K=K)

print(f"Item–item Precision@{K}: {p_item:.4f}, Recall@{K}: {r_item:.4f}")
print(f"Popularity Precision@{K}: {p_pop:.4f}, Recall@{K}: {r_pop:.4f}")


Item–item Precision@10: 0.0292, Recall@10: 0.2788
Popularity Precision@10: 0.0206, Recall@10: 0.2000
